## Curso 1: Creando tu primer Agent con Amazon Bedrock

## Preparacion 
<p style="padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Accede a <code>requirements.txt</code>, <code>helper.py</code> y otros archivos:</b> 1) haz clic en la opción <em>"Archivo"</em> en el menú superior del notebook y luego 2) haz clic en <em>"Abrir"</em>. Para más ayuda, consulta la lección <em>"Apéndice - Consejos y Ayuda"</em>.</p>

In [31]:
#  Setup del entorno
!sh ./shared/reset.sh

from dotenv import load_dotenv

load_dotenv()
import os

roleArn = os.environ["BEDROCKAGENTROLE"]

Resetting environment (if nessesary)
Found: agente_ventas_de_zapatos
Deleting alias: AgentTestAlias (ID: TSTALIASID)
Deletion initiated for alias: AgentTestAlias
Alias AgentTestAlias has been successfully deleted.
Deleting alias: MyAgentAlias (ID: ZLGFDUSJPF)
Deletion initiated for alias: MyAgentAlias
Alias MyAgentAlias has been successfully deleted.
Deleting agent: agente_ventas_de_zapatos (ID: I75WOQRHMI)
Deletion initiated for agent: agente_ventas_de_zapatos
Waiting for agent agente_ventas_de_zapatos to be deleted...
Agent agente_ventas_de_zapatos has been successfully deleted.
Agent reset process completed.
Lambda reset process completed.
Guardrail reset process completed.
Environment reset complete.


## Empezando la clase

In [14]:
# importando librerias
import boto3

In [15]:
# Cliente de Bedrock
bedrock_agent = boto3.client(service_name="bedrock-agent", region_name="us-east-1")

In [16]:
# Creamos el agente
create_agent_response = bedrock_agent.create_agent(
    agentName="agente_ventas_de_zapatos",
    foundationModel="anthropic.claude-3-haiku-20240307-v1:0",
    instruction="""Eres un agente de ventas especializado en calzado. Tu objetivo es ayudar a los clientes a encontrar los zapatos ideales según sus necesidades y preferencias.  """,
    agentResourceRoleArn=roleArn,
)

In [17]:
print(create_agent_response)

{'ResponseMetadata': {'RequestId': '91aca471-7f92-4341-9abb-4525f5da52e4', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Tue, 18 Mar 2025 15:33:41 GMT', 'content-type': 'application/json', 'content-length': '658', 'connection': 'keep-alive', 'x-amzn-requestid': '91aca471-7f92-4341-9abb-4525f5da52e4', 'x-amz-apigw-id': 'HoPNbFqnoAMEc5g=', 'x-amzn-trace-id': 'Root=1-67d99255-5a8761d32fb6e4464db3ad5a'}, 'RetryAttempts': 0}, 'agent': {'agentArn': 'arn:aws:bedrock:us-east-1:737993632905:agent/I75WOQRHMI', 'agentCollaboration': 'DISABLED', 'agentId': 'I75WOQRHMI', 'agentName': 'agente_ventas_de_zapatos', 'agentResourceRoleArn': 'arn:aws:iam::737993632905:role/BedrockAgentRole', 'agentStatus': 'CREATING', 'createdAt': datetime.datetime(2025, 3, 18, 15, 33, 41, 714970, tzinfo=tzutc()), 'foundationModel': 'anthropic.claude-3-haiku-20240307-v1:0', 'idleSessionTTLInSeconds': 600, 'instruction': 'Eres un agente de ventas especializado en calzado. Tu objetivo es ayudar a los clientes a encontrar 

In [18]:
agentId = create_agent_response["agent"]["agentId"]
print(agentId)
# guardar agentId en el .env

I75WOQRHMI


In [19]:
from shared.helper import *

In [20]:
wait_for_agent_status(agentId=agentId, targetStatus="NOT_PREPARED")

Waiting for agent status of 'NOT_PREPARED'...
Agent status: NOT_PREPARED
Agent reached 'NOT_PREPARED' status.


In [21]:
bedrock_agent.prepare_agent(agentId=agentId)

{'ResponseMetadata': {'RequestId': '546d3c17-1064-4586-8f9a-d80a9a10da69',
  'HTTPStatusCode': 202,
  'HTTPHeaders': {'date': 'Tue, 18 Mar 2025 15:33:51 GMT',
   'content-type': 'application/json',
   'content-length': '119',
   'connection': 'keep-alive',
   'x-amzn-requestid': '546d3c17-1064-4586-8f9a-d80a9a10da69',
   'x-amz-apigw-id': 'HoPO9GMTIAMEPmQ=',
   'x-amzn-trace-id': 'Root=1-67d9925f-7e85a2441a3f01ae0db9517b'},
  'RetryAttempts': 0},
 'agentId': 'I75WOQRHMI',
 'agentStatus': 'PREPARING',
 'agentVersion': 'DRAFT',
 'preparedAt': datetime.datetime(2025, 3, 18, 15, 33, 51, 557496, tzinfo=tzutc())}

In [22]:
wait_for_agent_status(agentId=agentId, targetStatus="PREPARED")

Waiting for agent status of 'PREPARED'...
Agent status: PREPARED
Agent reached 'PREPARED' status.


In [23]:
# Creando agent ALias
create_agent_alias_response = bedrock_agent.create_agent_alias(
    agentId=agentId,
    agentAliasName="MyAgentAlias",
)

agentAliasId = create_agent_alias_response["agentAlias"]["agentAliasId"]
# guardar agentAliasId en el .env

wait_for_agent_alias_status(
    agentId=agentId, agentAliasId=agentAliasId, targetStatus="PREPARED"
)

Waiting for agent alias status of 'PREPARED'...
Agent alias status: CREATING
Agent alias status: CREATING
Agent alias status: PREPARED
Agent alias reached status 'PREPARED'


In [24]:
bedrock_agent_runtime = boto3.client(
    service_name="bedrock-agent-runtime", region_name="us-east-1"
)

In [25]:
import uuid

In [26]:
# Preguntando al agente
message = "Hola, buenas tardes. Compré un zapato ayer, se rompió y quiero un reembolso."
sessionId = str(uuid.uuid4())

invoke_agent_response = bedrock_agent_runtime.invoke_agent(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    endSession=False,
    enableTrace=True,
)

In [27]:
# Agarrando streaming
event_stream = invoke_agent_response["completion"]

In [28]:
for event in event_stream:
    print(event)

{'trace': {'agentAliasId': 'ZLGFDUSJPF', 'agentId': 'I75WOQRHMI', 'agentVersion': '1', 'callerChain': [{'agentAliasArn': 'arn:aws:bedrock:us-east-1:737993632905:agent-alias/I75WOQRHMI/ZLGFDUSJPF'}], 'eventTime': datetime.datetime(2025, 3, 18, 15, 34, 30, 537181, tzinfo=tzutc()), 'sessionId': 'fa92fad8-3890-4b9e-8c13-55163d5ee5dd', 'trace': {'orchestrationTrace': {'modelInvocationInput': {'inferenceConfiguration': {'maximumLength': 2048, 'stopSequences': ['</invoke>', '</answer>', '</error>'], 'temperature': 0.0, 'topK': 250, 'topP': 1.0}, 'text': '{"system":" Eres un agente de ventas especializado en calzado. Tu objetivo es ayudar a los clientes a encontrar los zapatos ideales según sus necesidades y preferencias.   You have been provided with a set of functions to answer the user\'s question. You must call the functions in the format below: <function_calls>   <invoke>     <tool_name>$TOOL_NAME</tool_name>     <parameters>       <$PARAMETER_NAME>$PARAMETER_VALUE</$PARAMETER_NAME>      

In [29]:
# Nueva conversación
sessionId = str(uuid.uuid4())

In [30]:
# Funcion que creamos para que se entienda mejor los eventos
invoke_agent_and_print(
    agentAliasId=agentAliasId,
    agentId=agentId,
    sessionId=sessionId,
    inputText=message,
    enableTrace=True,
)

User: Hola, buenas tardes. Compré un zapato ayer, se rompió y quiero un
reembolso.

Agent: 
Agent's thought process:
  Entiendo que el cliente ha comprado un zapato que se rompió y ahora
  quiere un reembolso. Para poder ayudarlo, necesito obtener más
  información sobre el producto y la situación.

Agent's thought process:
  Disculpe, parece que hubo un error en la forma en que intenté
  invocar la función para obtener los detalles del producto. Voy a
  volver a intentarlo siguiendo el formato correcto.

Agent's thought process:
  Disculpe, parece que todavía no he logrado invocar la función
  correctamente. Voy a revisar cuidadosamente el formato requerido
  antes de intentarlo de nuevo.

Agent's thought process:
  Disculpe, parece que todavía no he logrado invocar la función
  correctamente. Voy a revisar cuidadosamente el formato requerido
  antes de intentarlo de nuevo.

Observation:
  Type: FINISH

Final response:
  Según la información proporcionada, el zapato que usted compró e